<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/languages/python/mini_projects/experiment_difference_guessing_game_cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Difference Guessing Game

This game challenges you to identify the item that doesn't belong to the same category as the others. You'll be presented with five options: four from one category and one from a different category. Your goal is to pick the 'odd one out'!

In [16]:
import random
import collections # To store previously asked questions

#### Game Resources

Here are the lists of items we'll be using for the game categories: Fruits, Cars, Vegetables, Countries, and Languages. Each list has 30 unique items.

In [17]:
# Define resource lists
fruits = [
    "Apple", "Banana", "Orange", "Grape", "Strawberry", "Blueberry", "Raspberry",
    "Pineapple", "Mango", "Kiwi", "Peach", "Plum", "Cherry", "Lemon", "Lime",
    "Avocado", "Pear", "Watermelon", "Cantaloupe", "Honeydew", "Fig", "Date",
    "Pomegranate", "Coconut", "Papaya", "Guava", "Lychee", "Dragonfruit", "Passionfruit", "Apricot"
]

cars = [
    "Toyota", "Honda", "Ford", "Chevrolet", "BMW", "Mercedes-Benz", "Audi",
    "Volkswagen", "Nissan", "Hyundai", "Kia", "Subaru", "Mazda", "Porsche",
    "Ferrari", "Lamborghini", "Tesla", "Volvo", "Jeep", "Ram", "GMC", "Lexus",
    "Acura", "Infiniti", "Chrysler", "Dodge", "Cadillac", "Buick", "Mitsubishi", "Suzuki"
]

vegetables = [
    "Carrot", "Broccoli", "Spinach", "Potato", "Tomato", "Onion", "Garlic",
    "Bell Pepper", "Cucumber", "Zucchini", "Lettuce", "Kale", "Cabbage", "Cauliflower",
    "Asparagus", "Green Bean", "Pea", "Corn", "Eggplant", "Pumpkin", "Sweet Potato",
    "Mushroom", "Celery", "Radish", "Artichoke", "Beet", "Squash", "Turnip", "Okra", "Brussels Sprout"
]

countries = [
    "USA", "Canada", "Mexico", "Brazil", "Argentina", "UK", "France",
    "Germany", "Italy", "Spain", "China", "India", "Japan", "Australia",
    "South Africa", "Egypt", "Nigeria", "Kenya", "Russia", "Turkey", "Saudi Arabia",
    "Sweden", "Norway", "Finland", "Denmark", "Netherlands", "Belgium", "Switzerland", "Austria", "Greece"
]

languages = [
    "English", "Spanish", "Mandarin", "Hindi", "French", "Arabic", "Bengali",
    "Russian", "Portuguese", "Urdu", "German", "Japanese", "Korean", "Italian",
    "Turkish", "Dutch", "Polish", "Vietnamese", "Javanese", "Punjabi", "Thai",
    "Gujarati", "Malay", "Telugu", "Tamil", "Marathi", "Cantonese", "Yoruba", "Hausa", "Burmese"
]

# Group all resources into a dictionary for easy access
categories = {
    "fruits": fruits,
    "cars": cars,
    "vegetables": vegetables,
    "countries": countries,
    "languages": languages
}

#### Helper Function: `shuffle_options`

This function takes a list of options and shuffles them, then assigns alphabetical labels (a, b, c, d, e) to each. This makes it easy to present multiple-choice questions to the user.

In [18]:
def shuffle_options(options):
    random.shuffle(options)
    labeled_options = {chr(ord('a') + i): option for i, option in enumerate(options)}
    return labeled_options

#### Game Question Generation Function: `generate_question`

This function is responsible for creating a single question for the game. It randomly selects a 'correct' category, picks four items from it, then chooses a different category and selects one item from it to be the 'odd one out'. It also ensures that the same question (defined by the combination of the main category and the odd one out item) is not repeated within a game session. If all possible unique questions are exhausted, it will indicate that the game is over.

In [19]:
asked_questions = set() # To store tuples of (main_category, odd_one_out_item) to prevent repetition

def generate_question():
    global asked_questions

    all_categories = list(categories.keys())

    # Loop to ensure unique questions
    attempts = 0
    max_attempts = 100 # Prevent infinite loops if questions run out

    while attempts < max_attempts:
        # 1. Choose a main category (where 4 items will come from)
        main_category_name = random.choice(all_categories)
        main_category_items = list(categories[main_category_name]) # Make a copy to sample from

        # 2. Choose an 'odd one out' category (must be different from main category)
        other_categories_names = [name for name in all_categories if name != main_category_name]
        if not other_categories_names:
            # This case should ideally not happen with 5 categories
            print("Error: Not enough categories to pick an 'odd one out'.")
            return None, None, None, None, None

        odd_one_out_category_name = random.choice(other_categories_names)
        odd_one_out_items = list(categories[odd_one_out_category_name]) # Make a copy

        # 3. Select 4 unique items from the main category
        if len(main_category_items) < 4:
            print(f"Warning: Not enough items in {main_category_name} for a question. Skipping this category.")
            attempts += 1
            continue
        correct_options = random.sample(main_category_items, 4)

        # 4. Select 1 unique item from the odd one out category
        if not odd_one_out_items:
            print(f"Warning: No items in {odd_one_out_category_name} for an odd one out. Skipping.")
            attempts += 1
            continue

        odd_one_out_item = random.choice(odd_one_out_items)

        # Check if this specific question has been asked before
        question_identifier = (main_category_name, odd_one_out_item)
        if question_identifier not in asked_questions:
            asked_questions.add(question_identifier)
            break
        attempts += 1
    else:
        # If max_attempts reached, implies all possible unique questions are exhausted
        print("All possible unique questions have been asked!")
        return None, None, None, None, None # Indicate no new question can be generated

    # Combine all options and shuffle them
    all_options_list = correct_options + [odd_one_out_item]
    labeled_options = shuffle_options(all_options_list)

    # Determine the correct answer label (key for the odd_one_out_item)
    correct_answer_label = None
    for label, item in labeled_options.items():
        if item == odd_one_out_item:
            correct_answer_label = label
            break

    return labeled_options, correct_answer_label, main_category_name, odd_one_out_item, odd_one_out_category_name


#### Main Game Function: `play_game`

This function orchestrates the entire game. It initializes the score, runs a set number of questions, uses the `generate_question` function to create new questions, processes user input, provides feedback, and asks the user if they want to play again.

In [20]:
def play_game(num_questions=5):
    global asked_questions
    total_score = 0
    questions_asked_in_round = 0
    print("\n--- Starting a new round of the Difference Guessing Game! ---")
    print(f"You will answer {num_questions} questions. Correct answers: +4 points, Incorrect answers: -1 point.\n")

    while questions_asked_in_round < num_questions:
        labeled_options, correct_answer_label, main_category_name, odd_one_out_item, odd_one_out_category_name = generate_question()

        if labeled_options is None: # No more unique questions
            print("We've run out of unique questions! Game over for now.")
            break

        questions_asked_in_round += 1
        print(f"\n--- Question {questions_asked_in_round}/{num_questions} ---")
        print(f"Which one is the odd one out?")
        for label, item in labeled_options.items():
            print(f"  {label.upper()}: {item}")

        while True:
            user_input = input("Your choice (A, B, C, D, E) or 'quit' to end: ").strip().lower()
            if user_input == 'quit':
                print("Exiting game. Thanks for playing!")
                return # Exit the game entirely

            # Handle variations like 'a.', 'b.'
            user_answer = user_input.replace('.', '')

            if user_answer in labeled_options:
                break
            else:
                print("Invalid input. Please enter A, B, C, D, E, or 'quit'.")

        if user_answer == correct_answer_label:
            total_score += 4
            print(f"Correct! '{odd_one_out_item}' is a {odd_one_out_category_name.capitalize()} while the others are {main_category_name.capitalize()}. Your score: {total_score}")
        else:
            total_score -= 1
            print(f"Incorrect. '{labeled_options[user_answer]}' is not the odd one out. The odd one out was '{odd_one_out_item}' (a {odd_one_out_category_name.capitalize()}). The others were {main_category_name.capitalize()}. Your score: {total_score}")

    print("\n--- Round Over! ---")
    print(f"Your final score for this round: {total_score}")

    while True:
        play_again = input("Do you want to play another round? (yes/no): ").strip().lower()
        if play_again in ['y', 'yes']:
            # Reset asked questions for a new full game experience, or keep them to ensure overall uniqueness
            asked_questions.clear()
            play_game(num_questions) # Start a new round recursively
            break # Exit this round's play_again loop
        elif play_again in ['n', 'no']:
            print("Thanks for playing! Goodbye!")
            break
        else:
            print("Please answer 'yes' or 'no'.")

# To start the game, simply call play_game()


### Ready to Play!

Run the cell below to start the **Difference Guessing Game**!

In [21]:
play_game()


--- Starting a new round of the Difference Guessing Game! ---
You will answer 5 questions. Correct answers: +4 points, Incorrect answers: -1 point.


--- Question 1/5 ---
Which one is the odd one out?
  A: Ford
  B: Thai
  C: Punjabi
  D: Malay
  E: Korean
Your choice (A, B, C, D, E) or 'quit' to end: a
Correct! 'Ford' is a Cars while the others are Languages. Your score: 4

--- Question 2/5 ---
Which one is the odd one out?
  A: Date
  B: Raspberry
  C: Lychee
  D: Bell Pepper
  E: Papaya
Your choice (A, B, C, D, E) or 'quit' to end: d
Correct! 'Bell Pepper' is a Vegetables while the others are Fruits. Your score: 8

--- Question 3/5 ---
Which one is the odd one out?
  A: Cauliflower
  B: Polish
  C: Russian
  D: Urdu
  E: Malay
Your choice (A, B, C, D, E) or 'quit' to end: e
Incorrect. 'Malay' is not the odd one out. The odd one out was 'Cauliflower' (a Vegetables). The others were Languages. Your score: 7

--- Question 4/5 ---
Which one is the odd one out?
  A: Volvo
  B: Mazda
  C